# Lenguajes y Autómatas II (SCD-1016)
## Sesión 03 · Google Colab, Mágicos y Flujo de Entrega

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo sabe una máquina de compilación si un paso terminó bien o falló?
>
> **Objetivo de la sesión:** Escribir y leer archivos desde C y devolver códigos de salida POSIX, para que un pipeline de compilación pueda encadenar pasos y detenerse ante un error.

**Entregable:** tus funciones en artesano_sesion03_c.c, el harness en verde y la bitácora de la sesión

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Tu programa debe escribir un archivo de texto, contar sus líneas y ejecutar un comando externo devolviendo su código de salida real, no el estado interno que entrega system().

Al final de la sesión tendrás: **Tres funciones en C que escriben y leen archivos y que reportan el código de salida real de un comando externo.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 03](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion03.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion03.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion03.h"

int escribir_texto(const char *ruta, const char *texto);
int contar_lineas(const char *ruta);
int ejecutar(const char *comando);
```

- Usa fopen/fputs o fgets y cierra siempre el archivo.
- Un archivo inexistente se reporta con -1, no con un aborto.
- El código de salida se extrae con WEXITSTATUS.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion03_c.c` junto con `artesano_sesion03_c.c`
> (o `ia_sesion03_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `escribir_texto(...)==0` | 0 |
| 2 | `contar_lineas(3 lineas)` | 3 |
| 3 | `contar_lineas(sin salto final)` | 2 |
| 4 | `contar_lineas(inexistente)` | -1 |
| 5 | `ejecutar("true")` | 0 |
| 6 | `ejecutar("false")` | 1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 03 - Archivos y codigos de salida ===
Caso                               esperado:              obtenido:              resultado
[ 1] escribir_texto(...)==0             esperado: 0            obtenido: 0            OK
[ 2] contar_lineas(3 lineas)            esperado: 3            obtenido: 3            OK
[ 3] contar_lineas(sin salto final)     esperado: 2            obtenido: 2            OK
[ 4] contar_lineas(inexistente)         esperado: -1           obtenido: -1           OK
[ 5] ejecutar("true")                   esperado: 0            obtenido: 0            OK
[ 6] ejecutar("false")                  esperado: 1            obtenido: 1            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Los archivos se cierran en todos los caminos y un error se reporta con -1.
- [ ] contar_lineas distingue un archivo con salto final de uno sin él.
- [ ] ejecutar devuelve 0 para un comando exitoso y 1 para uno fallido.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion03_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion03.h
/* api_sesion03.h - contrato de la Sesion 03
 * Archivos y códigos de salida
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION03_H
#define API_SESION03_H

int escribir_texto(const char *ruta, const char *texto);
int contar_lineas(const char *ruta);
int ejecutar(const char *comando);

#endif /* API_SESION03_H */


Writing api_sesion03.h


In [3]:
%%writefile artesano_sesion03_c.c
/* artesano_sesion03_c.c - Fase 1: El Artesano (sin IA)
 * Archivos y códigos de salida.
 *
 * Implementa lo que declara api_sesion03.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion03.h"

#include <stdio.h>
#include <stdlib.h>
#include <sys/wait.h>

int escribir_texto(const char *ruta, const char *texto) {
    FILE *f = fopen(ruta, "w");

    if (f == NULL) {
        return -1;
    }

    if (fputs(texto, f) == EOF) {
        fclose(f);
        return -1;
    }

    fclose(f);
    return 0;
}

int contar_lineas(const char *ruta) {
    FILE *f = fopen(ruta, "r");

    if (f == NULL) {
        return -1;
    }

    int c;
    int lineas = 0;
    int ultimo = '\n';
    int vacio = 1;

    while ((c = fgetc(f)) != EOF) {
        vacio = 0;

        if (c == '\n') {
            lineas++;
        }

        ultimo = c;
    }

    fclose(f);

    if (!vacio && ultimo != '\n') {
        lineas++;
    }

    return lineas;
}

int ejecutar(const char *comando) {
    int estado = system(comando);

    if (estado == -1) {
        return -1;
    }

    if (WIFEXITED(estado)) {
        return WEXITSTATUS(estado);
    }

    return -1;
}


Writing artesano_sesion03_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion03_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.

##C-R-E-O

**Contexto:**

 Estoy desarrollando una biblioteca en C para manejo de archivos y códigos de salida POSIX. Existe un archivo de cabecera llamado api_sesion03.h que declara tres funciones y sus firmas no pueden modificarse. El código será compilado junto con un harness usando gcc -std=c11 -Wall -Wextra -Werror.

**Rol:**

 Actúa como un desarrollador senior de C experto en manejo de archivos, entrada/salida estándar y procesos POSIX.

**Especificación:**
 Implementa exactamente las siguientes funciones sin agregar main():

```C
int escribir_texto(const char *ruta, const char *texto);
int contar_lineas(const char *ruta);
int ejecutar(const char *comando);

```

**Requisitos:**

- Escribir_texto debe abrir el archivo indicado por ruta, escribir el contenido de texto, cerrar el archivo y devolver 0 si tuvo éxito o -1 ante error.

- Contar_lineas debe abrir el archivo y contar correctamente sus líneas.

- Si el archivo no existe debe devolver -1.

- Debe distinguir entre un archivo cuyo contenido termina con salto de línea y otro que no.

- Ejecutar debe ejecutar el comando mediante system().

- Debe devolver el código de salida real del proceso usando WEXITSTATUS.
Si ocurre un error debe devolver -1.

- Todos los archivos abiertos deben cerrarse correctamente.

- Utiliza fopen, fputs, fgetc, fclose, system, WIFEXITED y WEXITSTATUS cuando corresponda.

**Output:**

 Devuelve únicamente código C compilable, respetando exactamente las firmas dadas, incluyendo los #include necesarios y sin agregar explicaciones ni texto adicional.

In [4]:
%%writefile ia_sesion03_c.c
/* ia_sesion03_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion03.h. No puede traer main().
 */
#include "api_sesion03.h"
#include <stdio.h>
#include <stdlib.h>
#include <sys/wait.h>

int escribir_texto(const char *ruta, const char *texto) {
    FILE *f = fopen(ruta, "w");

    if (f == NULL) {
        return -1;
    }

    if (fputs(texto, f) == EOF) {
        fclose(f);
        return -1;
    }

    fclose(f);
    return 0;
}

int contar_lineas(const char *ruta) {
    FILE *f = fopen(ruta, "r");

    if (f == NULL) {
        return -1;
    }

    int c;
    int lineas = 0;
    int ultimo = '\n';
    int vacio = 1;

    while ((c = fgetc(f)) != EOF) {
        vacio = 0;

        if (c == '\n') {
            lineas++;
        }

        ultimo = c;
    }

    fclose(f);

    if (!vacio && ultimo != '\n') {
        lineas++;
    }

    return lineas;
}

int ejecutar(const char *comando) {
    int estado = system(comando);

    if (estado == -1) {
        return -1;
    }

    if (WIFEXITED(estado)) {
        return WEXITSTATUS(estado);
    }

    return -1;
}

Writing ia_sesion03_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion03_c.c
/* harness_sesion03_c.c - auditoria automatica de la Sesion 03
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion03_c.c artesano_sesion03_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion03.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 03 - Archivos y codigos de salida ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = escribir_texto("datos.txt", "uno\ndos\ntres\n"); verificar_int("escribir_texto(...)==0", (long)(0), (long)(r)); }
    { escribir_texto("datos.txt", "uno\ndos\ntres\n"); verificar_int("contar_lineas(3 lineas)", (long)(3), (long)(contar_lineas("datos.txt"))); }
    { escribir_texto("datos2.txt", "a\nb"); verificar_int("contar_lineas(sin salto final)", (long)(2), (long)(contar_lineas("datos2.txt"))); }
    {  verificar_int("contar_lineas(inexistente)", (long)(-1), (long)(contar_lineas("no_existe.txt"))); }
    {  verificar_int("ejecutar(\"true\")", (long)(0), (long)(ejecutar("true"))); }
    {  verificar_int("ejecutar(\"false\")", (long)(1), (long)(ejecutar("false"))); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md" "\n");
    return 0;
}


Writing harness_sesion03_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion03_c.c artesano_sesion03_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 03 - Archivos y codigos de salida ===
Caso                               esperado:              obtenido:              resultado
[ 1] escribir_texto(...)==0             esperado: 0            obtenido: 0            OK
[ 2] contar_lineas(3 lineas)            esperado: 3            obtenido: 3            OK
[ 3] contar_lineas(sin salto final)     esperado: 2            obtenido: 2            OK
[ 4] contar_lineas(inexistente)         esperado: -1           obtenido: -1           OK
[ 5] ejecutar("true")                   esperado: 0            obtenido: 0            OK
[ 6] ejecutar("false")                  esperado: 1            obtenido: 1            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion03_c.c ia_sesion03_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 03 - Archivos y codigos de salida ===
Caso                               esperado:              obtenido:              resultado
[ 1] escribir_texto(...)==0             esperado: 0            obtenido: 0            OK
[ 2] contar_lineas(3 lineas)            esperado: 3            obtenido: 3            OK
[ 3] contar_lineas(sin salto final)     esperado: 2            obtenido: 2            OK
[ 4] contar_lineas(inexistente)         esperado: -1           obtenido: -1           OK
[ 5] ejecutar("true")                   esperado: 0            obtenido: 0            OK
[ 6] ejecutar("false")                  esperado: 1            obtenido: 1            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion03.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion03.md` a `auditorias/auditoria_sesion03.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
